In [ ]:
# @title 1. Setup and Connect
import sqlite3
import pandas as pd
import os
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import drive
from google.colab import files

db_path = '/content/CRdatabase.db'
print('Downloading CRdatabase.db...')
!wget -q 'https://raw.githubusercontent.com/edear-dev/CRAB/main/CRdatabase.db'
print('Database downloaded successfully.')


Database downloaded successfully.


In [ ]:
# @title 2. Load Helper Functions
def list_options(column_name):
    """Fetches unique values for dropdown menus."""
    valid_cols = ['element', 'exp_name', 'year']
    if column_name not in valid_cols: return []

    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        query = f"SELECT DISTINCT {column_name} FROM experiments ORDER BY {column_name}"
        df = pd.read_sql_query(query, conn)
        return df[column_name].tolist()
    except:
        return []
    finally:
        conn.close()

def get_data(element=None, experiment=None, data_type='E'):
    """Main search function handling the table joins."""
    table_map = {
        'E': ('energies', 'E'),
        'E_per_n': ('energy_per_n', 'E_per_n'),
        'R': ('rigidities', 'R')
    }

    child_table, x_col = table_map[data_type]

    query = f"""
        SELECT
            e.exp_name, e.year, e.element,
            c.{x_col} as {data_type},
            c.flux, c.error, c.flux_unit
        FROM experiments e
        JOIN {child_table} c ON e.exp_id = c.exp_id
        WHERE 1=1
    """

    params = []
    if element and element != 'All':
        query += " AND e.element = ?"
        params.append(element)
    if experiment and experiment != 'All':
        query += " AND e.exp_name = ?"
        params.append(experiment)

    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        return pd.read_sql_query(query, conn, params=params)
    finally:
        conn.close()

print("Functions loaded.")

Functions loaded.


In [ ]:
# @title 3. Search Interface
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files

# Get valid experiment, element combos
def get_combinations():
    conn = sqlite3.connect(f"file:{db_path}?mode=ro", uri=True)
    try:
        query = "SELECT DISTINCT element, exp_name FROM experiments"
        return pd.read_sql_query(query, conn)
    finally:
        conn.close()

df_combos = get_combinations()
all_elements = sorted(df_combos['element'].unique().tolist())
all_experiments = sorted(df_combos['exp_name'].unique().tolist())

# setup the widgets
style = {'description_width': 'initial'}
w_header = widgets.HTML("<h3>Cosmic Ray Database Search</h3>")

w_elem = widgets.Dropdown(options=['All'] + all_elements, value='All', description='Element:', style=style)
w_exp = widgets.Dropdown(options=['All'] + all_experiments, value='All', description='Experiment:', style=style)
w_type = widgets.Dropdown(options=[('Total Energy', 'E'), ('Energy / Nucleon', 'E_per_n'), ('Rigidity', 'R')], description='Data Type:', style=style)

# buttons
btn_search = widgets.Button(description="Search Data", button_style='primary', icon='search')
btn_download = widgets.Button(description="Download CSV", button_style='success', icon='download', disabled=True)
btn_reset = widgets.Button(description="Reset Options", button_style='warning', icon='refresh')

output_area = widgets.Output()
current_search_result = None

# changes options without changing widget value
def set_options_silently(widget, new_options, current_value):
    widget.unobserve(update_menus, names='value')
    widget.options = new_options
    if current_value in new_options:
        widget.value = current_value
    else:
        widget.value = 'All'
    widget.observe(update_menus, names='value')

# handles menu changes, updates menus accordingly
def update_menus(change):
    owner = change['owner']
    new_val = change['new']

    if owner == w_elem:
        current_exp = w_exp.value
        if new_val == 'All':
            new_opts = ['All'] + all_experiments
        else:
            valid_exps = df_combos[df_combos['element'] == new_val]['exp_name'].unique().tolist()
            valid_exps.sort()
            new_opts = ['All'] + valid_exps
        set_options_silently(w_exp, new_opts, current_exp)

    elif owner == w_exp:
        current_elem = w_elem.value
        if new_val == 'All':
            new_opts = ['All'] + all_elements
        else:
            valid_elems = df_combos[df_combos['exp_name'] == new_val]['element'].unique().tolist()
            valid_elems.sort()
            new_opts = ['All'] + valid_elems
        set_options_silently(w_elem, new_opts, current_elem)

    # Disable download on any change
    btn_download.disabled = True
    global current_search_result
    current_search_result = None
    with output_area: clear_output()

# resets the options
def on_reset_click(b):
    """
    Forces all widgets back to their starting state.
    """
    # Unplug the observers so we don't trigger the update logic 4 times in a row
    w_elem.unobserve(update_menus, names='value')
    w_exp.unobserve(update_menus, names='value')

    # Reset everything to default
    w_elem.options = ['All'] + all_elements
    w_elem.value = 'All'

    w_exp.options = ['All'] + all_experiments
    w_exp.value = 'All'

    # Clear Output
    btn_download.disabled = True
    global current_search_result
    current_search_result = None
    with output_area: clear_output()

    # Plug the observers back in
    w_elem.observe(update_menus, names='value')
    w_exp.observe(update_menus, names='value')

def on_search_click(b):
    global current_search_result
    with output_area:
        clear_output()
        print("Searching...")
        s_elem = w_elem.value if w_elem.value != 'All' else None
        s_exp = w_exp.value if w_exp.value != 'All' else None
        df = get_data(element=s_elem, experiment=s_exp, data_type=w_type.value)

        if df.empty:
            print("No data found.")
            btn_download.disabled = True
        else:
            current_search_result = df
            btn_download.disabled = False
            print(f"Found {len(df)} rows.")
            display(df.head(10))

def on_download_click(b):
    if current_search_result is not None:
        filename = f"CRDB_{w_elem.value}_{w_exp.value}.csv"
        current_search_result.to_csv(filename, index=False)
        files.download(filename)

# bind functions
w_elem.observe(update_menus, names='value')
w_exp.observe(update_menus, names='value')
w_type.observe(update_menus, names='value')

btn_search.on_click(on_search_click)
btn_download.on_click(on_download_click)
btn_reset.on_click(on_reset_click)

# Layout
ui_top = widgets.HBox([w_elem, w_exp, w_type])
ui_buttons = widgets.HBox([btn_search, btn_download, btn_reset])

display(widgets.VBox([w_header, ui_top, ui_buttons, output_area]))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>